In [1]:
import os, re, pypdf, torch, numpy as np, pandas as pd
from tqdm import tqdm
import nltk
nltk.data.path.append('C:/nltk_data')
from nltk.tokenize import sent_tokenize
from transformers import BertTokenizer, AutoModelForSequenceClassification

DATA_DIR = "."
OUTPUT_DIR = os.path.join(DATA_DIR, "Output_Results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

nltk.download('punkt', quiet=True)
DEVICE = torch.device("cpu")
print("Loading FinBERT model (first time may download ~500MB)...")
tokenizer = BertTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert").to(DEVICE)
model.eval()
print("Model loaded.")

def extract_sentences(pdf_path):
    text = ""
    for page in pypdf.PdfReader(pdf_path).pages:
        text += page.extract_text() or ""
    text = re.sub(r'\s+', ' ', text)
    return [s for s in sent_tokenize(text) if len(s.split()) >= 5]

def analyze_sentiment(sentences, batch_size=16):
    label_map = {0: "positive", 1: "negative", 2: "neutral"}
    results = []
    for i in tqdm(range(0, len(sentences), batch_size), desc="Processing"):
        batch = sentences[i:i+batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors="pt")
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            probs = torch.softmax(model(**inputs).logits, dim=-1).cpu().numpy()
            preds = np.argmax(probs, axis=-1)
        for j, pred in enumerate(preds):
            results.append({
                "sentence": batch[j],
                "sentiment": label_map[pred],
                "positive_score": probs[j][0],
                "negative_score": probs[j][1],
                "neutral_score": probs[j][2]
            })
    return pd.DataFrame(results)

pdf_files = [f for f in os.listdir(DATA_DIR) if f.lower().endswith('.pdf')]
print(f"Found {len(pdf_files)} PDF files.")

for f in pdf_files:
    output_csv = os.path.join(OUTPUT_DIR, f.replace('.pdf', '.csv'))
    if os.path.exists(output_csv):
        print(f"Skipping {f} (already processed).")
        continue
    path = os.path.join(DATA_DIR, f)
    print(f"\nProcessing {f} ...")
    sentences = extract_sentences(path)
    if len(sentences) == 0:
        print("  No valid sentences found, skipping.")
        continue
    print(f"  Extracted {len(sentences)} sentences.")
    df = analyze_sentiment(sentences)
    pos = (df['sentiment'] == 'positive').mean()
    neg = (df['sentiment'] == 'negative').mean()
    neu = (df['sentiment'] == 'neutral').mean()
    print(f"  Positive: {pos:.2%} | Negative: {neg:.2%} | Neutral: {neu:.2%}")
    df.to_csv(output_csv, index=False, encoding='utf-8-sig')
    print(f"  Results saved to {output_csv}")

print("\nAll done!")

Loading FinBERT model (first time may download ~500MB)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded.
Found 25 PDF files.
Skipping E.ON_FY2025_results_20260225.pdf.pdf (already processed).
Skipping E.ON_2017_GB_EN_.pdf.pdf (already processed).
Skipping E.ON_2023_GB_engl_gesamt_final.pdf.pdf (already processed).
Skipping E.ON_2015_financial_statements.pdf.pdf (already processed).
Skipping E.ON_2016_financial_statements.pdf.pdf (already processed).
Skipping E.ON_2017_JA_EN.pdf.pdf (already processed).
Skipping E.ONSE_JA_2023_en.pdf.pdf (already processed).
Skipping E.ON_20260225_capital_markets_story_final.pdf.pdf (already processed).
Skipping E.ON_2014_annual_report.pdf.pdf (already processed).
Skipping E.ON_2015_annual_report.pdf.pdf (already processed).
Skipping E.ON_2020_annual_report.pdf.pdf (already processed).
Skipping E.ON_2021_EA21_ESE21_US.pdf.pdf (already processed).
Skipping E.ON_2022_JA_LB_TA_en.pdf.pdf (already processed).
Skipping E.ON_2018_GB_US_final.pdf.pdf (already processed).
Skipping E.ON_2019_GB_US_final.pdf.pdf (already processed).
Skipping E.ON_2020_

In [ ]:
import os
print(os.listdir("."))

['E.ON_FY2025_results_20260225.pdf.pdf', 'E.ON_2015_segment_information.xlsx.xlsx', 'E.ON_2015_group_key_figures.xlsx.xlsx', 'E.ON_2016_segment_information.xlsx.xlsx', 'E.ON_2016_group_key_figures.xlsx.xlsx', 'E.ON_2017_GB_EN_.pdf.pdf', 'E.ON_FY2020_financial_info_by_business_segment.xlsx.xlsx', 'E.ON_2023_GB_engl_gesamt_final.pdf.pdf', 'E.ON_2015_financial_statements.pdf.pdf', 'E.ON_2016_financial_statements.pdf.pdf', 'E.ON_2017_JA_EN.pdf.pdf', 'E.ONSE_JA_2023_en.pdf.pdf', 'E.ON_20260225_capital_markets_story_final.pdf.pdf', 'E.ON_FY2025_results_20260225.xlsx.xlsx', 'E.ON_20200226_tables.xlsx.xlsx', 'E.ON_20260225_capital_markets_story_final.xlsx.xlsx', 'E.ON_FY2020_group_financial_highlights.xlsx.xlsx', 'E.ON_2018_Q4_group_financial_highlights.xlsx.xlsx', 'E.ON_2017_Q4_group_financial_highlights.xlsx.xlsx', 'E.ON_2019_Q4_group_financial_highlights.xlsx.xlsx', 'E.ON_2018_Q4_information_by_business_segment.xlsx.xlsx', 'E.ON_2014_annual_report.pdf.pdf', 'E.ON_2015_annual_report.pdf.pdf'